# SuperSpellCheck: fine-tune the on-device AI checker

Fine-tunes **Qwen3-0.6B** into a minimal-edit sentence corrector and exports it as a 4-bit ONNX model the extension runs in the browser. Works on a free **Colab T4** or **Kaggle 2×T4**. See `train/README.md` for the step-by-step guide.

**Before running:**
- **Colab:** Runtime → Change runtime type → **T4 GPU**.
- **Kaggle:** Settings → Accelerator → **GPU T4 x2**, and turn **Internet** on.
- Add a Hugging Face **write** token as a secret named `HF_TOKEN` (Colab: the key icon in the sidebar; Kaggle: Add-ons → Secrets).
- If the GitHub repo is private, also add a `GITHUB_TOKEN` secret.

In [ ]:
# 1. Settings: edit these
HF_REPO = "YOUR_HF_USERNAME/superspellcheck-qwen3-0.6b-gec-ONNX"  # where the browser model is published
N_EXAMPLES = 300_000      # training pairs; 300k takes ~2-4 h on a T4
GITHUB_REPO = "andrewnakas/superspellcheck"
BRANCH = "claude/friendly-wozniak-vwfrjk"

In [ ]:
# 2. Detect the platform and fetch secrets
import os, subprocess, sys
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
def secret(name):
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        try: return UserSecretsClient().get_secret(name)
        except Exception: return None
    if ON_COLAB:
        from google.colab import userdata
        try: return userdata.get(name)
        except Exception: return None
    return os.environ.get(name)
HF_TOKEN, GH_TOKEN = secret("HF_TOKEN"), secret("GITHUB_TOKEN")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
print("kaggle" if ON_KAGGLE else "colab" if ON_COLAB else "local", "| HF token:", bool(HF_TOKEN))
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. On Colab, keep checkpoints on Google Drive so a disconnect doesn't lose progress
OUT = f"{WORK}/ssc-out"
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/ssc-out"
os.makedirs(OUT, exist_ok=True)
print("checkpoints ->", OUT)

In [ ]:
# 4. Get the code and install dependencies (~3 min)
%cd {WORK}
url = f"https://{GH_TOKEN}@github.com/{GITHUB_REPO}.git" if GH_TOKEN else f"https://github.com/{GITHUB_REPO}.git"
if not os.path.exists("superspellcheck"):
    !git clone -q --depth 1 -b {BRANCH} {url} superspellcheck
%cd superspellcheck
!pip install -q "transformers==4.57.1" "trl==0.24.0" "peft==0.17.1" "datasets==4.2.0" "accelerate==1.10.1" pyarrow \
    "onnx==1.19.1" "onnxruntime==1.23.2" "onnxruntime-genai==0.11.4" onnx-ir huggingface_hub

In [ ]:
# 5. (Recommended) Build the Rust engine: make_data.py uses it to drop training targets that still contain errors (~4 min)
!curl -sSf https://sh.rustup.rs | sh -s -- -y -q --profile minimal > /dev/null
!$HOME/.cargo/bin/cargo build --release -q && ls -la target/release/ssc

In [ ]:
# 6. Build the training set (downloads one 371 MB C4_200M shard)
!python train/make_data.py --n {N_EXAMPLES} --out {OUT}/data --cache {WORK}/cache
!head -c 600 {OUT}/data/train.jsonl

In [ ]:
# 7. Fine-tune (resumes automatically from the last checkpoint if re-run)
import torch
n_gpu = torch.cuda.device_count()
if n_gpu > 1:
    !accelerate launch --num_processes {n_gpu} --mixed_precision fp16 train/finetune.py --data {OUT}/data --out {OUT}/run --batch 16 --grad-accum 1
else:
    !python train/finetune.py --data {OUT}/data --out {OUT}/run --batch 16 --grad-accum 2

In [ ]:
# 8. Evaluate (grammar test set, clean-sentence false alarms)
!bash bench/fetch.sh > /dev/null
!python bench/ai_eval.py --model {OUT}/run/merged

In [ ]:
# 9. Export to 4-bit ONNX for the browser (~5 min)
!python train/export.py --merged {OUT}/run/merged --out {OUT}/onnx-model
!ls -la {OUT}/onnx-model {OUT}/onnx-model/onnx

In [ ]:
# 10. Publish to the Hugging Face Hub
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.create_repo(HF_REPO, exist_ok=True)
open(f"{OUT}/onnx-model/README.md", "w").write(f"""---
license: apache-2.0
base_model: Qwen/Qwen3-0.6B
library_name: transformers.js
tags: [grammatical-error-correction, onnx]
---
Qwen3-0.6B fine-tuned by SuperSpellCheck as a minimal-edit English sentence corrector, exported to 4-bit ONNX (q4f16) for transformers.js.
Training data: C4_200M (CC BY 4.0) plus synthetic errors. Prompt format: train/prompt.json in the SuperSpellCheck repo.
""")
api.upload_folder(folder_path=f"{OUT}/onnx-model", repo_id=HF_REPO)
print(f"Published: https://huggingface.co/{HF_REPO}")
print(f"Next: set aiModel to '{HF_REPO}' in extension/src/types.ts (DEFAULT_SETTINGS)")